# UNet Baseline — BUSI Breast Ultrasound Segmentation

**Purpose:** Train a standard 2D UNet as a baseline to compare against SuperLightNet2D.  
All training hyperparameters are kept **identical** to the SuperLightNet2D run.

| Setting | Value |
|---|---|
| Dataset | BUSI (binary segmentation) |
| Image size | 256 × 256 |
| Epochs | 200 |
| Batch size | 8 |
| Optimizer | AdamW (lr=1e-3, wd=1e-2) |
| LR Schedule | Cosine Annealing (T_max=50) |
| Loss | BCEWithDiceLoss |
| AMP | ✅ |


## 0. Setup

In [1]:
import os, sys, time, random, logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Point to project root so we can reuse existing modules
PROJECT_ROOT = os.path.abspath('')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dataset import BUSIDataset, BUSI_MEAN, BUSI_STD
from losses  import BCEWithDiceLoss
from metrics import compute_all_metrics, dice_coefficient, iou_score, \
                    hausdorff_distance_95, sensitivity, specificity, precision

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Using device : cuda
GPU          : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM         : 6.4 GB


## 1. Configuration — same as SuperLightNet2D run

In [2]:
# Paths
DATA_DIR   = r'C:\Prudhvi\sem7\MIP\SuperLightNet\BUSI_SuperLightNet\BUSI_Dataset\Dataset_BUSI_with_GT'
EXPERIMENT = 'unet_baseline'
CKPT_DIR   = os.path.join('checkpoints', EXPERIMENT)
LOG_DIR    = os.path.join('logs', EXPERIMENT)
EVAL_DIR   = os.path.join('eval_output', EXPERIMENT)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# Hyperparameters (identical to SuperLightNet2D)
IMG_SIZE     = 256
BATCH_SIZE   = 8
NUM_WORKERS  = 4
EPOCHS       = 200
LR           = 1e-3
WEIGHT_DECAY = 1e-2
T_MAX        = 50
ETA_MIN      = LR * 0.1
SAVE_FREQ    = 10
SEED         = 2024
AMP          = torch.cuda.is_available()

# UNet channel widths
N_CHANNELS = 32
INIT_CH    = 1
CLASS_NUMS = 2

# Reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('Config ready.')
print(f'AMP enabled: {AMP}')

Config ready.
AMP enabled: True


## 2. Standard 2D UNet Architecture

In [3]:
class DoubleConv(nn.Module):
    """Two consecutive Conv2d -> BatchNorm -> ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class Down(nn.Module):
    """MaxPool2d(2) -> DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.block(x)


class Up(nn.Module):
    """Bilinear x2 upsample -> concat skip -> DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x, skip):
        x  = self.up(x)
        dh = skip.size(2) - x.size(2)
        dw = skip.size(3) - x.size(3)
        x  = F.pad(x, [dw//2, dw - dw//2, dh//2, dh - dh//2])
        return self.conv(torch.cat([skip, x], dim=1))


class UNet2D(nn.Module):
    """
    Standard 2D UNet.
    Input:  [B, 1,  256, 256]
    Output: [B, 2,  256, 256]
    Channel widths (n=32): 32->64->128->256->512 (bottleneck)
    """
    def __init__(self, in_ch=1, n=32, class_nums=2):
        super().__init__()
        self.enc0       = DoubleConv(in_ch, n)
        self.enc1       = Down(n,    n*2)
        self.enc2       = Down(n*2,  n*4)
        self.enc3       = Down(n*4,  n*8)
        self.bottleneck = Down(n*8,  n*16)
        self.dec3       = Up(n*16 + n*8, n*8)
        self.dec2       = Up(n*8  + n*4, n*4)
        self.dec1       = Up(n*4  + n*2, n*2)
        self.dec0       = Up(n*2  + n,   n)
        self.head       = nn.Conv2d(n, class_nums, 1)

    def forward(self, x):
        s0 = self.enc0(x)
        s1 = self.enc1(s0)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        x  = self.bottleneck(s3)
        x  = self.dec3(x, s3)
        x  = self.dec2(x, s2)
        x  = self.dec1(x, s1)
        x  = self.dec0(x, s0)
        return self.head(x)


# Sanity check
_m     = UNet2D(INIT_CH, N_CHANNELS, CLASS_NUMS)
_dummy = torch.rand(2, 1, 256, 256)
_out   = _m(_dummy)
_p     = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'UNet2D   Input : {list(_dummy.shape)}')
print(f'UNet2D   Output: {list(_out.shape)}')
print(f'UNet2D   Params: {_p:.3f} M')
print('SuperLightNet2D (small): ~4.1 M  (for reference)')

UNet2D   Input : [2, 1, 256, 256]
UNet2D   Output: [2, 2, 256, 256]
UNet2D   Params: 7.849 M
SuperLightNet2D (small): ~4.1 M  (for reference)


## 3. Data Loaders

In [4]:
train_ds = BUSIDataset(os.path.join(DATA_DIR, 'train.txt'), DATA_DIR, 'train', IMG_SIZE)
val_ds   = BUSIDataset(os.path.join(DATA_DIR, 'val.txt'),   DATA_DIR, 'val',   IMG_SIZE)
test_ds  = BUSIDataset(os.path.join(DATA_DIR, 'test.txt'),  DATA_DIR, 'test',  IMG_SIZE)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

print(f'Train : {len(train_ds)} samples  ({len(train_loader)} batches)')
print(f'Val   : {len(val_ds)} samples')
print(f'Test  : {len(test_ds)} samples')

Train : 545 samples  (68 batches)
Val   : 115 samples
Test  : 120 samples


## 4. Model, Optimizer, Scheduler, Loss

In [5]:
model     = UNet2D(INIT_CH, N_CHANNELS, CLASS_NUMS).to(DEVICE)
criterion = BCEWithDiceLoss().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=ETA_MIN)
scaler    = torch.amp.GradScaler('cuda', enabled=AMP)

params_M = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model on : {DEVICE}')
print(f'Params   : {params_M:.3f} M')
print(f'AMP      : {AMP}')

Model on : cuda
Params   : 7.849 M
AMP      : True


## 5. Training Loop

In [6]:
# Logging
for h in logging.root.handlers[:]: logging.root.removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(os.path.join(LOG_DIR, 'train.log'), mode='w')
    ]
)

history = {'train_loss': [], 'train_dice': [], 'val_loss': [],
           'val_dsc': [], 'val_iou': [], 'val_hd95': []}


def train_one_epoch(epoch):
    model.train()
    total_loss = total_dice = 0.0
    for step, (images, masks) in enumerate(train_loader, 1):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=AMP):
            logits = model(images)
            loss, dice = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 12)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        total_dice += dice.item()
    scheduler.step()
    return total_loss / len(train_loader), total_dice / len(train_loader)


@torch.no_grad()
def validate():
    model.eval()
    total_loss = 0.0
    all_m = {k: [] for k in ('DSC', 'IoU', 'HD95', 'Sens', 'Spec', 'Prec')}
    for images, masks in val_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=AMP):
            logits = model(images)
            loss, _ = criterion(logits, masks)
        total_loss += loss.item()
        m = compute_all_metrics(logits, masks)
        for k in all_m: all_m[k].append(m[k])
    avg = {k: float(np.nanmean(v)) for k, v in all_m.items()}
    return total_loss / len(val_loader), avg


# ─── Main loop ────────────────────────────────────────────────────────────────
best_dsc = 0.0
t_start  = time.time()
logging.info(f'Starting UNet baseline training — {EPOCHS} epochs on {DEVICE}')

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    tr_loss, tr_dice = train_one_epoch(epoch)
    val_loss, val_m  = validate()
    val_dsc          = val_m['DSC']

    lr_now = optimizer.param_groups[0]['lr']
    ep_min = (time.time() - t_ep) / 60
    eta_h  = (EPOCHS - epoch) * ep_min / 60

    history['train_loss'].append(tr_loss)
    history['train_dice'].append(tr_dice)
    history['val_loss'].append(val_loss)
    history['val_dsc'].append(val_dsc)
    history['val_iou'].append(val_m['IoU'])
    history['val_hd95'].append(val_m['HD95'])

    logging.info(
        f'Ep {epoch:03d}/{EPOCHS} | LR={lr_now:.2e} | '
        f'TrLoss={tr_loss:.4f} | TrDice={tr_dice:.4f} | '
        f'ValLoss={val_loss:.4f} | ValDSC={val_dsc:.4f} | '
        f'ValIoU={val_m["IoU"]:.4f} | {ep_min:.1f}min | ETA={eta_h:.2f}h'
    )

    if val_dsc > best_dsc:
        best_dsc = val_dsc
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pth'))
        logging.info(f'  *** Best DSC={best_dsc:.4f} — saved best_model.pth ***')

    if epoch % SAVE_FREQ == 0:
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, f'epoch_{epoch:03d}.pth'))

total_h = (time.time() - t_start) / 3600
logging.info(f'Done. Best Val DSC={best_dsc:.4f} | Total={total_h:.2f}h')
print(f'\nBest model -> {CKPT_DIR}/best_model.pth')

13:44:35 | Starting UNet baseline training — 200 epochs on cuda
c:\Prudhvi\sem7\MIP\SuperLightNet\BUSI_SuperLightNet\metrics.py:114: RuntimeWarning: Mean of empty slice
  return {k: float(np.nanmean(v)) for k, v in results.items()}
13:45:03 | Ep 001/200 | LR=9.99e-04 | TrLoss=1.2871 | TrDice=0.1352 | ValLoss=1.3315 | ValDSC=0.1575 | ValIoU=0.0986 | 0.5min | ETA=1.59h
13:45:04 |   *** Best DSC=0.1575 — saved best_model.pth ***
13:45:32 | Ep 002/200 | LR=9.96e-04 | TrLoss=1.0289 | TrDice=0.2437 | ValLoss=1.9909 | ValDSC=0.1985 | ValIoU=0.1270 | 0.5min | ETA=1.57h
13:45:32 |   *** Best DSC=0.1985 — saved best_model.pth ***
13:46:36 | Ep 003/200 | LR=9.92e-04 | TrLoss=0.8650 | TrDice=0.3647 | ValLoss=1.0585 | ValDSC=0.3565 | ValIoU=0.3012 | 1.1min | ETA=3.49h
13:46:36 |   *** Best DSC=0.3565 — saved best_model.pth ***
13:47:02 | Ep 004/200 | LR=9.86e-04 | TrLoss=0.8211 | TrDice=0.4073 | ValLoss=0.8746 | ValDSC=0.5090 | ValIoU=0.4179 | 0.4min | ETA=1.44h
13:47:02 |   *** Best DSC=0.5090 — s


Best model -> checkpoints\unet_baseline/best_model.pth


## 6. Training Curves

In [7]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('UNet Baseline — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_x, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_x, history['val_loss'],   label='Val',   color='tomato')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_dsc'], color='seagreen')
axes[1].axhline(best_dsc, color='gold', ls='--', label=f'Best={best_dsc:.4f}')
axes[1].set_title('Val DSC'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, history['val_iou'], color='mediumpurple')
axes[2].set_title('Val IoU'); axes[2].grid(alpha=0.3)

plt.tight_layout()
out_path = os.path.join(EVAL_DIR, 'training_curves.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')

Saved -> eval_output\unet_baseline\training_curves.png


C:\Users\gprud\AppData\Local\Temp\ipykernel_38844\3653643424.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Test Set Evaluation

In [8]:
state = torch.load(os.path.join(CKPT_DIR, 'best_model.pth'), map_location='cpu')
model.load_state_dict(state)
model.eval()
print('Loaded best_model.pth')

vis_dir = os.path.join(EVAL_DIR, 'visualizations')
os.makedirs(vis_dir, exist_ok=True)

keys       = ('DSC', 'IoU', 'HD95', 'Sens', 'Spec', 'Prec')
per_image  = {k: [] for k in keys}
res_lines  = ['idx,' + ','.join(keys)]
global_idx = 0


def save_panel(img_t, gt, pred, path):
    img = (img_t.squeeze().numpy() * BUSI_STD + BUSI_MEAN).clip(0, 1)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, title, ov, cm in zip(axes,
            ['Input', 'Ground Truth', 'Prediction'],
            [None, gt, pred], [None, 'Reds', 'Blues']):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        if ov is not None:
            ax.imshow(ov.astype(float), alpha=0.45, cmap=cm, vmin=0, vmax=1)
        ax.set_title(title, fontsize=11); ax.axis('off')
    plt.tight_layout()
    plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.close(fig)


with torch.no_grad():
    for images, masks in test_loader:
        logits = model(images.to(DEVICE))
        preds  = (logits.argmax(dim=1) == 1).cpu().numpy()
        tgts   = masks.numpy()
        for b in range(images.size(0)):
            p, t = preds[b], tgts[b]
            m = {'DSC': dice_coefficient(p, t), 'IoU': iou_score(p, t),
                 'HD95': hausdorff_distance_95(p, t), 'Sens': sensitivity(p, t),
                 'Spec': specificity(p, t), 'Prec': precision(p, t)}
            for k in keys: per_image[k].append(m[k])
            res_lines.append(f'{global_idx},' + ','.join(f'{m[k]:.4f}' for k in keys))
            if global_idx < 20:
                save_panel(images[b], t, p,
                           os.path.join(vis_dir, f'pred_{global_idx:04d}.png'))
            global_idx += 1
        print(f'\r  {global_idx}/{len(test_ds)}', end='', flush=True)
print()

unet_results = {}
print('\n' + '='*55)
print('UNET BASELINE — TEST RESULTS')
print('='*55)
for k in keys:
    vals = [v for v in per_image[k] if not np.isnan(v)]
    mean_, std_ = np.mean(vals), np.std(vals)
    unet_results[k] = mean_
    print(f'  {k:5s}: {mean_:.4f} +/- {std_:.4f}')
    res_lines.append(f'MEAN_{k},{mean_:.4f}')
print('='*55)

with open(os.path.join(EVAL_DIR, 'results.txt'), 'w') as f:
    f.write('\n'.join(res_lines) + '\n')
print(f'\nResults  -> {EVAL_DIR}/results.txt')
print(f'Visuals  -> {vis_dir}/')

Loaded best_model.pth
  120/120

UNET BASELINE — TEST RESULTS
  DSC  : 0.7129 +/- 0.3344
  IoU  : 0.6398 +/- 0.3358
  HD95 : 39.2504 +/- 46.8890
  Sens : 0.7838 +/- 0.3001
  Spec : 0.9861 +/- 0.0297
  Prec : 0.7834 +/- 0.3125

Results  -> eval_output\unet_baseline/results.txt
Visuals  -> eval_output\unet_baseline\visualizations/


## 8. Sample Predictions (inline)

In [9]:
fig, axes = plt.subplots(6, 3, figsize=(13, 24))
fig.suptitle('UNet Baseline — Sample Test Predictions', fontsize=13, fontweight='bold')
sample_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
model.eval()
with torch.no_grad():
    for i, (img, msk) in enumerate(sample_loader):
        if i >= 6: break
        logit = model(img.to(DEVICE))
        pred  = (logit.argmax(dim=1) == 1).squeeze().cpu().numpy()
        gt    = msk.squeeze().numpy()
        disp  = (img.squeeze().numpy() * BUSI_STD + BUSI_MEAN).clip(0, 1)
        dsc_i = dice_coefficient(pred, gt)
        axes[i][0].imshow(disp, cmap='gray'); axes[i][0].set_title('Input'); axes[i][0].axis('off')
        axes[i][1].imshow(disp, cmap='gray')
        axes[i][1].imshow(gt.astype(float),   alpha=0.5, cmap='Reds')
        axes[i][1].set_title('Ground Truth'); axes[i][1].axis('off')
        axes[i][2].imshow(disp, cmap='gray')
        axes[i][2].imshow(pred.astype(float), alpha=0.5, cmap='Blues')
        axes[i][2].set_title(f'Pred  DSC={dsc_i:.3f}'); axes[i][2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'sample_predictions.png'), dpi=100, bbox_inches='tight')
plt.show()

C:\Users\gprud\AppData\Local\Temp\ipykernel_38844\1870058176.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Final Comparison — UNet vs SuperLightNet2D

In [10]:
# Load SuperLightNet2D results if available
sln_path    = os.path.join('eval_output', 'busi_v1', 'results.txt')
sln_results = {}
if os.path.exists(sln_path):
    with open(sln_path) as f:
        for line in f:
            for k in keys:
                if line.startswith(f'MEAN_{k}'):
                    sln_results[k] = float(line.strip().split(',')[1])
    print('Loaded SuperLightNet2D results from eval_output/busi_v1/results.txt')
else:
    print('SuperLightNet2D results not found — using README expected ranges as placeholder.')
    sln_results = {'DSC':0.76,'IoU':0.65,'HD95':17.5,'Sens':0.80,'Spec':0.97,'Prec':0.78}

# Print table
unet_params = sum(p.numel() for p in model.parameters()) / 1e6
print()
print('='*70)
print(f'{"Metric":>8}  {"UNet2D":>18}  {"SuperLightNet2D":>18}  {"Winner":>12}')
print('-'*70)
higher_better = {'DSC','IoU','Sens','Spec','Prec'}
for k in keys:
    u = unet_results.get(k, float('nan'))
    s = sln_results.get(k, float('nan'))
    if k in higher_better:
        winner = 'SLN' if s > u + 0.005 else ('UNet' if u > s + 0.005 else 'Tie')
    else:
        winner = 'SLN' if s < u - 0.5 else ('UNet' if u < s - 0.5 else 'Tie')
    print(f'{k:>8}  {u:>18.4f}  {s:>18.4f}  {winner:>12}')
print('='*70)
print(f'{"Params":>8}  {unet_params:>18.3f}M  {4.095:>17.3f}M')
print()

# Save CSV
comp = ['Metric,UNet2D,SuperLightNet2D']
for k in keys:
    comp.append(f'{k},{unet_results.get(k,0):.4f},{sln_results.get(k,0):.4f}')
comp.append(f'Params_M,{unet_params:.3f},4.095')
with open(os.path.join(EVAL_DIR, 'comparison.csv'), 'w') as f:
    f.write('\n'.join(comp) + '\n')
print(f'Comparison CSV -> {EVAL_DIR}/comparison.csv')

Loaded SuperLightNet2D results from eval_output/busi_v1/results.txt

  Metric              UNet2D     SuperLightNet2D        Winner
----------------------------------------------------------------------
     DSC              0.7129              0.6613          UNet
     IoU              0.6398              0.5839          UNet
    HD95             39.2504             31.4896           SLN
    Sens              0.7838              0.7506          UNet
    Spec              0.9861              0.9849           Tie
    Prec              0.7834              0.7307          UNet
  Params               7.849M              4.095M

Comparison CSV -> eval_output\unet_baseline/comparison.csv


## 10. Comparison Bar Chart

In [11]:
plot_keys = ['DSC', 'IoU', 'Sens', 'Spec', 'Prec']
unet_vals = [unet_results.get(k, 0) for k in plot_keys]
sln_vals  = [sln_results.get(k, 0)  for k in plot_keys]
x, w      = np.arange(len(plot_keys)), 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('UNet2D vs SuperLightNet2D — BUSI Test Set', fontsize=14, fontweight='bold')

# Main metrics
ax = axes[0]
b1 = ax.bar(x - w/2, unet_vals, w, label='UNet2D',          color='steelblue',  alpha=0.85)
b2 = ax.bar(x + w/2, sln_vals,  w, label='SuperLightNet2D', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(plot_keys)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Main Metrics (higher = better)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in [*b1, *b2]:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=7)

# HD95
ax2 = axes[1]
hd  = [unet_results.get('HD95',0), sln_results.get('HD95',0)]
bs  = ax2.bar(['UNet2D','SuperLightNet2D'], hd,
               color=['steelblue','darkorange'], alpha=0.85, width=0.4)
ax2.set_ylabel('Pixels'); ax2.set_title('HD95 (lower = better)')
ax2.grid(axis='y', alpha=0.3)
for b in bs:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{b.get_height():.2f}px', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'comparison_chart.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved comparison chart.')

Saved comparison chart.


C:\Users\gprud\AppData\Local\Temp\ipykernel_38844\1304653961.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Summary of Outputs

| File | Path |
|---|---|
| Best weights | `checkpoints/unet_baseline/best_model.pth` |
| Training log | `logs/unet_baseline/train.log` |
| Test metrics | `eval_output/unet_baseline/results.txt` |
| Comparison CSV | `eval_output/unet_baseline/comparison.csv` |
| Training curves | `eval_output/unet_baseline/training_curves.png` |
| Sample predictions | `eval_output/unet_baseline/sample_predictions.png` |
| Comparison chart | `eval_output/unet_baseline/comparison_chart.png` |